# Hafta 11 — Colab: benchmark + eğitimler

Çalışma zamanı → GPU. Bu çıktılar **A100 40GB**'den (T4 istedik, Colab A100 verdi; `benchmark.py` 20 GB üstü GPU'da B=16 kullanıyor, T4'te B=4). VS Code Colab eklentisi: Explorer'da `week11/week11_colab.zip`'e sağ tık → *Upload to Colab*, sonra hücreleri sırayla çalıştır.

Zip'in içinde: `week11/` (gpt2, dataloader, data, train, benchmark), `week08/input.txt`, `week08/turkce.txt`, `week09/bpe.py`, `week09/bpe_cache.pt`. Zip'i oluşturmak için repo kökünde:
```
zip week11/week11_colab.zip week11/{gpt2.py,dataloader.py,data.py,train.py,benchmark.py} week08/input.txt week08/turkce.txt week09/bpe.py week09/bpe_cache.pt
```

In [2]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
%cd /content
!unzip -oq week11_colab.zip && mkdir -p week11/samples && ls week08 week09 week11
import torch; print(torch.__version__)

name, memory.total [MiB], compute_cap
NVIDIA A100-SXM4-40GB, 40960 MiB, 8.0
/content
week08:
input.txt  turkce.txt

week09:
bpe_cache.pt  bpe.py

week11:
benchmark.py  dataloader.py  data.py  gpt2.py  samples	train.py
2.11.0+cu130


## Görev 2: hızlandırma tablosu (~10 dk)
fp32 → TF32 → autocast (T4: fp16 + GradScaler) → torch.compile → flash → vocab 50,304, sonra her adım tek başına. B=4, T=1024.

In [3]:
%cd /content/week11
!python benchmark.py 2>&1 | tee samples/benchmark_cuda.txt | grep -v '  adım'

/content/week11
cihaz: NVIDIA A100-SXM4-40GB | B=16, T=1024 (16,384 token/adım)
[0] fp32 (başlangıç)
    1038 ms/adım, 15,790 token/sn  
[1] + TF32
    378 ms/adım, 43,291 token/sn  
[2] + bf16 / fp16 autocast
    341 ms/adım, 48,063 token/sn  
[3] + torch.compile
    138 ms/adım, 118,812 token/sn  
[4] + flash attention
    97 ms/adım, 169,180 token/sn  
[5] + vocab 50,304
    93 ms/adım, 176,188 token/sn  
[6] tek başına: autocast
    341 ms/adım, 48,076 token/sn  
[7] tek başına: torch.compile
    915 ms/adım, 17,914 token/sn  
[8] tek başına: flash attention
    774 ms/adım, 21,172 token/sn  
[9] tek başına: vocab 50,304
    1037 ms/adım, 15,792 token/sn  
[10] compile'sız: autocast + flash + 50,304
    126 ms/adım, 130,525 token/sn  
[11] compile'sız, autocast'sız: flash + 50,304
    774 ms/adım, 21,173 token/sn  

| adım | ms/adım | token/sn | hızlanma | not |
|---|---:|---:|---:|---|
| fp32 (başlangıç) | 1038 | 15,790 | 1.00× |  |
| + TF32 | 378 | 43,291 | 2.74× |  |
| + bf16 / 

## Görev 1: data loader ile Shakespeare'in tamamında (~6 dk)
fp32, sabit lr 3e-4, düz AdamW. 1 epoch = 73 batch (4×1024), 300 adım ≈ 4 epoch.

In [4]:
!python train.py --name task1_shakespeare --B 4 --T 1024 --steps 300 --eval_every 25 2>&1 | tee samples/task1_shakespeare.txt | grep -E 'val|epoch [0-9]$' | awk 'NR%5==1' ; tail -4 samples/task1_shakespeare.txt

step    0 | val 10.9535
step    4 | loss 8.4051 | lr 3.00e-04 | norm   3.063 | 102 ms | 40,266 tok/s | epoch 0
step    9 | loss 7.3867 | lr 3.00e-04 | norm   2.117 | 101 ms | 40,367 tok/s | epoch 0
step   14 | loss 7.0366 | lr 3.00e-04 | norm   1.712 | 102 ms | 40,038 tok/s | epoch 0
step   19 | loss 6.7628 | lr 3.00e-04 | norm   1.182 | 102 ms | 40,318 tok/s | epoch 0
step   24 | loss 6.8247 | lr 3.00e-04 | norm   0.901 | 102 ms | 40,274 tok/s | epoch 0
step   28 | loss 6.7657 | lr 3.00e-04 | norm   1.111 | 101 ms | 40,389 tok/s | epoch 0
step   33 | loss 6.6056 | lr 3.00e-04 | norm   0.867 | 102 ms | 40,248 tok/s | epoch 0
step   38 | loss 6.3880 | lr 3.00e-04 | norm   0.957 | 102 ms | 40,336 tok/s | epoch 0
step   43 | loss 6.2611 | lr 3.00e-04 | norm   1.046 | 102 ms | 40,331 tok/s | epoch 0
step   48 | loss 6.2124 | lr 3.00e-04 | norm   0.925 | 101 ms | 40,383 tok/s | epoch 0
step   52 | loss 6.5420 | lr 3.00e-04 | norm   1.126 | 102 ms | 40,338 tok/s | epoch 0
step   57 | loss 6.

## Görev 3: GPT-3 ayarları + gradient accumulation (~10 dk)
AdamW (0.9, 0.95), wd 0.1 (2D), clip 1.0, warmup 15 + cosine 150 adım, 32,768 token/adım = 8 × (4×1024). fp16 + flash + compile + vocab 50,304.

In [5]:
!python train.py --name task3_gpt3 --gpt3 --total_batch 32768 --B 4 --T 1024 --steps 150 --warmup 15 --autocast --flash --compile --eval_every 25 2>&1 | tee samples/task3_gpt3.txt | grep -E 'weight decay|step +[0-9]*[05] \|' ; tail -4 samples/task3_gpt3.txt

weight decay: 50 tensör, 124,354,560 parametre | decay yok: 98 tensör, 121,344 parametre | fused AdamW: True
step    0 | val 10.9534
step    0 | loss 10.9401 | lr 4.00e-05 | norm  29.875 | 19879 ms | 1,648 tok/s | epoch 0
step    5 | loss 8.8485 | lr 2.40e-04 | norm   3.416 | 217 ms | 150,998 tok/s | epoch 0
step   10 | loss 7.5930 | lr 4.40e-04 | norm   2.015 | 217 ms | 150,834 tok/s | epoch 1
step   15 | loss 6.4110 | lr 6.00e-04 | norm   1.330 | 217 ms | 150,947 tok/s | epoch 1
step   20 | loss 6.1336 | lr 5.98e-04 | norm   1.554 | 217 ms | 151,022 tok/s | epoch 2
step   25 | val 6.3430
step   25 | loss 6.3034 | lr 5.93e-04 | norm   1.448 | 219 ms | 149,638 tok/s | epoch 2
step   30 | loss 6.1366 | lr 5.84e-04 | norm   0.809 | 218 ms | 150,534 tok/s | epoch 3
step   35 | loss 5.9594 | lr 5.71e-04 | norm   0.826 | 218 ms | 150,508 tok/s | epoch 3
step   40 | loss 5.9630 | lr 5.56e-04 | norm   0.912 | 218 ms | 150,471 tok/s | epoch 4
step   45 | loss 5.9135 | lr 5.37e-04 | norm   0.71

## Ek (b): Nutuk, aynı model, aynı süre (45'er sn), iki tokenizer
GPT-2 tokenizer'ı (vocab 50,257 → 50,304) ve hafta 9 BPE (vocab 512). Cosine süre oranıyla iniyor.

İlk deneme 8'er dakikaydı (T4 hesabıyla seçilmişti). A100'de bu 120-147 epoch'a denk geldi, model metni ezberledi ve val loss en iyi noktadan sonra sürekli yükseldi. O koşunun val eğrileri `samples/extra_8dk_overfit.json`'da.

In [11]:
%cd /content/week11
A = "--gpt3 --B 4 --T 1024 --steps 1000000 --time_budget 45 --warmup 20 --autocast --flash --compile --eval_every 100 --eval_batches 13"
!python train.py --name extra_nutuk_gpt2 --data nutuk_gpt2 {A} 2>&1 | tail -4
!python train.py --name extra_nutuk_bpe512 --data nutuk_bpe512 {A} 2>&1 | tail -4

/content/week11

1301 adım, 40 sn eğitim | görülen token 5,328,896 (~10,416,712 karakter, 10.52 epoch)
tam val loss 3.7392 nats/token | 1.9334 nats/karakter
örnek: Efendiler, bu meplerini vak hakikleri çızuruna kalkında kuvvet, bizimle meşi beyanlar olduğunlaklara tespştına mülâtın hakâd-ı askerinci nakip ettiğizin taararınd

1706 adım, 40 sn eğitim | görülen token 6,987,776 (~13,412,773 karakter, 13.53 epoch)
tam val loss 4.6216 nats/token | 2.3923 nats/karakter
örnek: Efendiler, garen temîebusine ile kadarası yaptığc Eskiyeye’le İstanbul’diyor ve Ceple, Ali Kumandanı val-i meydada, Kuvan Vekili Harp olunanıp görülyla, teşkilât-ı millî’n


## Sonuçları Mac'e almak
Zip Colab sunucusunda oluşuyor, Mac'e kendiliğinden inmiyor. Bu hücre `samples/`'ı base64 metin olarak yazdırıyor; notebook kaydedilince çıktı yerel `.ipynb`'ye yazılıyor ve oradan çözülüyor (ya da Colab panelinden elle indir).

In [ ]:
import base64, zipfile, io, glob
buf = io.BytesIO()
with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob("/content/week11/samples/*"):
        z.write(f, "samples/" + f.split("/")[-1])
print("B64START" + base64.b64encode(buf.getvalue()).decode() + "B64END")